# 과제 2 — 요청서 1건으로 코호트 만들기 (20점)

- **하는 일**: 연구자의 요청서 1건을 SQL 로 옮깁니다. 조건을 한 줄씩 더하며 건수를 구하고(S1~S5), 동료 SQL 을 점검하고(E1~E3), 검산표를 만들어(T1 · T2) 해석합니다.
- **분석 질환**: 제2형 당뇨병 (`'Diabetes Mellitus, Type 2'`), 모든 학생 공통
- **제출**: **2026-12-08(화)** 까지, LMS 에 `과제2.ipynb`

| 부분 | 문항 | 배점 |
|---|---|---|
| 1. 단계별 추출 (11 · 13주차) | S1~S4 | 각 2점 (8점) |
| 2. 작은 칸 가리기 (13주차) | S5 | 2점 |
| 3. 동료 SQL 점검 (12주차) | E1~E3 | 각 1점 (3점) |
| 4. 검산표 만들기 (10주차) | T1 · T2 | 각 2점 (4점) |
| 5. 결과 해석 (13주차) | ✏️ 해석 | 3점 |
| 합계 | | 20점 |

S · E · T 문항은 ✅ 가 나오면 점수입니다. 해석은 교수자가 채점합니다.

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.
- ✏️ 칸은 두 번 눌러 `→` 뒤에 적습니다.

### 푸는 방법

1. S · T 문항은 `___` 를 채워 실행하고, E 문항은 동료 SQL 의 한 군데를 고쳐 실행합니다.
2. 결과 아래에 ✅ 가 나오면 점수입니다. ❌ 가 두 번 나오면 💡 힌트가 보입니다.

In [ ]:
#@title ✏️ 학번 · 이름을 입력하고 실행하세요 (준비 셀) { display-mode: "form" }
학번 = ""  #@param {type:"string"}
이름 = ""  #@param {type:"string"}
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

MESH = ["Diabetes Mellitus, Type 2"]
학번, 이름 = str(globals().get("학번", "")).strip(), str(globals().get("이름", "")).strip()
MY_MESH = MESH[int(_hl.md5(학번.encode()).hexdigest(), 16) % len(MESH)]      # 지금은 모든 학생 공통
if not 학번 or 학번 == "20260000":
    print("⚠️ 학번이 비어 있습니다. 위 칸에 학번과 이름을 입력하고 이 준비 셀을 다시 실행하세요.")
else:
    print(f"연결 완료.  {이름}({학번})")
print(f"분석 질환: {MY_MESH} (제2형 당뇨병, 모든 학생 공통)")

def 제출확인():
    """제출 전에 실행: 문항별 결과와 제출 방법을 보여 줍니다."""
    print(f"제출자: {이름 or '(이름 없음)'} ({학번 or '학번 없음'})  /  분석 질환: {MY_MESH}")
    진행확인()
    print("\n제출: ① Runtime → Run all (오류에서 멈추면 아래 셀을 누르고 Runtime → Run after)  ② File → Download → Download .ipynb  ③ LMS 에 과제2.ipynb 제출")

ANSWERS = _load_answers("eNq1lmFP20YYx7/Ko7wKEuocO07IpL5ANBJIFZ0I3V5MU5Um9opEoQKmaqoquatpDcmkZJDGoBClJTSwMckjBoIUrVI/SvrOd/4Oe+4cg22BRCrtDTn+5/ju/7v/81xexHKJ2LfwIra68DN+xmQ5JYt5JZ/IFOXYOMSWb9BXlp+vop7AYWF5kQ1/jNFSw62b1DBjP6H8ZGFpjX2Tdgx3xwS6Z5OPOv4FUtbomzI57tFmBZzzHv5P3jeAbLXpOxto0yCb20DrNm2uk5JBSu078FRZffJo7ddnCr5GA/+NH9bJgTkOxeXnS4X8qvLIe0pZeeo95a/BX25tO//2rzZB9vogAN/NbyY1T4cLcccrRWVFKeLW1fziqsKUZ2sLy0v5RV96iVpODFITJamYTxZSUiqZCVKL6iNQa+hA1zVwTiyn26d7JjjWH3QPxZMW4/b5/AaO40iwiv7JtkVN2602mH3HqtBSD2jrFd30vYL4+Vyir1/Fv5uezGXFb/iHNMbQEX2fdpBgbfPq8RHQSEE0aqGQKaSSgpQQikE0Uf32aAaa6Vg7uKeBtsOichfGwdOGm76aIlUk09M4oJ7lmr1Apu4COcBEbFnkyGZU3d0yqTTAW5N0ypiMw6+ynwzaF1IFWUqISkIRC0H7Uf06++RMJ38eslkcoYcQBW8SXdAdnewHth6wGAlRvUL1BkPlfQUZoQRuzSalC7dmMmB+yG4XoRGgyKFMyKqkpoV0Ri2GMiGooqooGTXzOJMOQEnJQSrO6d+OpbFpz4bnOsSGu/AbBDtlXv49ZtCtHWKeLhlBPCHS/T61DoGabXwaUkA6BgaJdDaAtnVWaHT3iHemVo18PIYvWtk1a0C6GjahL9rvY3fAsXXMFkbPYMtO5uD7ybmp6ck54EivOlz1/bXg1lZ+uYlbNtSc1UxKlYSJvCykw7UU0a8Lk4ctWkusaLDgvcnL9qghJtzuicHKgOo9WuujcuTHaurBw9n5oGlMEW1XSfeUZ6f+htZavPo6FSawbuJHBuITQ9z8JfF7M7n5mdmpedbvx8ZGCFQ21H8lScwLyUwinX4cAhPVb99kSOnQ6eoMCrkw/MpCMDM5mH14/z7w89y22AeWhNNtwYM5LtLm20CPnZy9hwWq8UnWbfQmJ1Ir82RwgvxNx2cBRGk/kc0KjmBYmrTUxk2CPBIl6f+9pWZmI9nHZsHsstsbfeYSMGw6dMtmc7ya6rZ/I3/a8C1nfMt6g/TK9FMfK20Uo/OhOkkLxaKgJCcSSlIKGo3qQ6NiqOnyg2ezzj921G8uy5Ja4cfx2gDviTg1/mI/M1ji/Y7Jq2DPHhuHH6azc1n/S5gTetEC990Gg+UtdVl1t267MKi2wLkos+sOMRKtTOsGbfT5RdbWyAXv+uRMc3ULF2nj2bH04KoYJVbOu9ujoA1VGv78S0+oyTw2byGINqoP0SZvjXZQ/YCXWQ7PYlDdZyOJjdpslAROi194TDwIiaxKAlfejWfxVVfXy/8A4oAA6A==")   # 자동 확인용 (정답 SQL 은 없음)

## 요청서

| 항목 | 내용 |
|---|---|
| 연구 질문 | 제2형 당뇨병 **3상** 임상시험 중 결과가 게시된 비율은 얼마인가? |
| 대상 (포함) | MeSH `'Diabetes Mellitus, Type 2'`, 임상 단계 3상 (2·3상을 함께 하는 시험은 넣지 않음) |
| 제외 | 상태가 철회(WITHDRAWN)인 시험 |
| 분모 · 분자 | 분모 = 완료(COMPLETED)된 시험, 분자 = 그중 결과게시일(`results_first_posted_date`)이 있는 시험 |
| 세는 단위 | 임상시험 1건 (`nct_id`, 중복 없이) |
| 결과 형태 | 검산표 1개 + 국가별 표 1개 (5건 미만인 칸은 '5 미만') |

13주차 수업 예시(완료된 제2형 당뇨병 전체 시험)와 같은 절차를 3상 시험에 적용합니다.

## 데이터 구성 — 어떤 표가 있고, 어떻게 연결되나

표 이름과 연결 키(FK)는 이 그림과 표에서 확인합니다.

![aact_mini 표 구조 한눈에 보기](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_erd.png)

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | studies 와의 관계 | 주로 쓰는 열 |
|---|---|---|---|---|---|
| `studies` | 임상시험 1건 | `nct_id` | — | 기준이 되는 표 | brief_title, study_type, overall_status, phase, start_date, enrollment, results_first_posted_date |
| `eligibilities` | 선정 · 제외 기준 | `id` | `nct_id` → studies | 1 : 1 (시험마다 1행) | gender, minimum_age (글자) |
| `sponsors` | 스폰서 1곳 | `id` | `nct_id` → studies | 1 : N (하나 이상) | agency_class, lead_or_collaborator, name |
| `conditions` | 대상 질환 1개 (연구자가 적은 이름) | `id` | `nct_id` → studies | 1 : N (하나 이상) | name (자유 입력이라 표기가 제각각) |
| `browse_conditions` | 표준 질환명(MeSH) 1개 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | mesh_term |
| `countries` | 참여 국가 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, removed (TRUE 면 등록 후 빠진 국가) |
| `facilities` | 수행 기관 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, country |

**읽는 법** (5 · 6주차)

1. **기본키** — 한 행을 하나로 찾는 열입니다. `studies` 는 `nct_id`(임상시험 등록번호), 나머지 표는 `id` 입니다.
2. **외래키** — 다른 표의 `nct_id` 는 `studies.nct_id` 를 가리킵니다. 그래서 표를 연결(JOIN)할 때는 늘 `ON studies.nct_id = 다른표.nct_id` 로 잇습니다.
3. **1 : N** — 시험 1건에 국가 · 기관 · 질환이 여러 개면 그 표에는 행이 여러 개입니다. 연결하면 시험 1건이 여러 행이 되므로, 시험 수는 `COUNT(DISTINCT studies.nct_id)` 로 셉니다. 또 `browse_conditions` · `countries` · `facilities` 에는 행이 하나도 없는 시험도 있어서, INNER JOIN 하면 그 시험이 빠집니다.

In [ ]:
%%sql
-- 확인: studies 에서는 1행인 시험이 countries 와 연결하면 국가 수만큼 여러 행이 됩니다
SELECT studies.nct_id, overall_status, name AS 국가
FROM studies
JOIN countries
  ON studies.nct_id = countries.nct_id
WHERE studies.nct_id = 'NCT01795521'

**열 이름 찾기** — 열 이름이 생각나지 않으면 표를 먼저 봅니다. 아래 셀에서 표 이름만 바꿔 실행하세요.

In [ ]:
%%sql
-- 열 이름 보기 (7주차 SELECT * — 전체 열 보기): studies 를 browse_conditions · countries 로 바꿔 실행
SELECT *
FROM studies
LIMIT 3;

## 1. 단계별 추출 — 조건을 한 줄씩 (11 · 13주차)

9주차 · 13주차처럼 한 단계에 WHERE 한 줄씩 더합니다. 앞 단계 줄은 이미 적혀 있고, 새 줄의 `___` 만 채웁니다. 줄 끝 주석(② ~ ⑥)은 4절 검산표의 단계입니다.

### S1 · 대상 질환 (2점)

요청서의 대상 질환을 WHERE 한 줄로 옮깁니다. 질환 이름이 들어 있는 **열 이름**을 채우세요.

💡 힌트: browse_conditions 의 열 이름은 바로 아래 DESCRIBE 결과에서 찾습니다 (11주차 데이터 사전).

In [ ]:
%%sql
-- browse_conditions 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE browse_conditions);

In [ ]:
%%sql
-- S1 (2점)
-- ___ 를 채워 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE ___ = 'Diabetes Mellitus, Type 2';   -- ② 제2형 당뇨

### S2 · 3상만 (2점)

요청서의 ‘3상’이 데이터에는 어떤 글자로 적혀 있는지 먼저 보고, 그 **값**을 채우세요. 2·3상을 함께 하는 시험은 넣지 않습니다.

💡 힌트: 값의 종류는 바로 아래 셀처럼 GROUP BY 로 봅니다 (12주차 「오류를 찾는 SQL 네 가지」 ② · 오류 9 — 대소문자와 공백).

In [ ]:
%%sql
-- phase 열에 어떤 값이 있는지 보기 (전체 시험)
SELECT phase, COUNT(*) AS 시험수
FROM studies
GROUP BY phase
ORDER BY phase;

In [ ]:
%%sql
-- S2 (2점)
-- ___ 를 채워 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'   -- ② 제2형 당뇨
  AND phase = ___;   -- ③ 3상

### S3 · 철회 제외 (2점)

철회(WITHDRAWN)된 시험을 뺍니다. ‘같지 않다’를 뜻하는 **비교 기호**를 채우세요.

💡 힌트: 7주차 「WHERE 조건에 쓰는 기호」.

In [ ]:
%%sql
-- S3 (2점)
-- ___ 를 채워 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'   -- ② 제2형 당뇨
  AND phase = 'PHASE3'   -- ③ 3상
  AND overall_status ___ 'WITHDRAWN';   -- ④ 철회 제외

### S4 · 분모와 분자 (2점)

SELECT 가 분모 · 분자 두 열로 바뀝니다 (13주차 「단계별 추출 — 9강 방식 그대로」). 분모가 되는 **완료** 상태 값을 채우세요. 분자 줄(CASE WHEN)은 적혀 있습니다.

💡 힌트: 상태 값의 종류는 바로 아래 셀에서 봅니다. S3 의 'WITHDRAWN' 처럼 대문자 영어입니다.

In [ ]:
%%sql
-- overall_status 열에 어떤 값이 있는지 보기 (전체 시험)
SELECT overall_status, COUNT(*) AS 시험수
FROM studies
GROUP BY overall_status
ORDER BY overall_status;

In [ ]:
%%sql
-- S4 (2점)
-- ___ 를 채워 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 분모,
       COUNT(DISTINCT CASE WHEN results_first_posted_date IS NOT NULL
                           THEN studies.nct_id END) AS 분자
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'   -- ② 제2형 당뇨
  AND phase = 'PHASE3'   -- ③ 3상
  AND overall_status <> 'WITHDRAWN'   -- ④ 철회 제외
  AND overall_status = ___;   -- ⑤ 완료 (분모) · ⑥ 결과 게시 (분자)

## 2. 작은 칸 가리기 (13주차)

결과표의 칸이 아주 작으면 그 칸은 가려서 내보냅니다 (11주차 「결과를 전달하는 형태」 · 13주차 「결과표를 내보내기 전에」).

### S5 · 국가별 표 — 작은 칸 가리기 (2점)

요청서의 두 번째 표입니다. 국가별 완료 시험 수에서 5건 미만인 칸은 '5 미만'으로 가립니다. 숫자와 글자를 한 열에 같이 쓰려면 숫자를 글자로 바꿔야 합니다. `___` 에 숫자를 글자로 바꾸는 **함수 이름**을 채우세요.

💡 힌트: 13주차 「작은 칸 가리기 — CASE WHEN 으로 ‘5 미만’」 · 12주차 용어 정리 「형 변환」.

In [ ]:
%%sql
-- S5 (2점)
-- ___ 를 채워 실행하세요
SELECT countries.name AS 국가,
       CASE WHEN COUNT(DISTINCT studies.nct_id) < 5 THEN '5 미만'
            ELSE ___(COUNT(DISTINCT studies.nct_id) AS VARCHAR)
       END AS 완료시험
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
JOIN countries
  ON studies.nct_id = countries.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND phase = 'PHASE3'
  AND overall_status = 'COMPLETED'
  AND removed = false
GROUP BY countries.name
ORDER BY COUNT(DISTINCT studies.nct_id) DESC, 국가;

## 3. 동료 SQL 점검 (12주차)

동료의 SQL 3개는 **오류 없이 실행되지만 숫자가 틀립니다**. 12주차 오류 체크리스트로 틀린 한 군데를 찾아 고치세요.

### E1 · 참여 국가 수 (1점)

동료가 원한 것: 완료된 3상 제2형 당뇨병 시험에 참여한 국가는 몇 곳인가요? (빠진 국가 removed 는 제외)

💡 힌트: 한 나라가 여러 시험에 참여하면 그 나라 행이 여러 개입니다. 무엇을 세야 하나요? (12주차 오류 1 — 세는 단위). 고친 숫자는 S5 표의 행 수와 같아야 합니다.

In [ ]:
%%sql
-- E1 (1점)
-- [고칠 SQL] 동료가 쓴 SQL — 오류 없이 실행되지만 숫자가 틀립니다. 한 군데를 고쳐 실행하세요
SELECT COUNT(countries.name) AS 국가수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
JOIN countries
  ON studies.nct_id = countries.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND phase = 'PHASE3'
  AND overall_status = 'COMPLETED'
  AND removed = false;

### E2 · 3상이 아닌 시험 수 (1점)

동료가 원한 것: 제2형 당뇨병 시험 중 3상이 **아닌** 시험(임상 단계가 비어 있는 시험 포함)은 몇 건인가요?

💡 힌트: 검산 — S1(제2형 당뇨 전체) = S2(3상) + E2 가 되어야 합니다. 빈 값(NULL)은 = · <> 로 비교하면 걸리지 않습니다 (7주차 자주 하는 실수 1 · 12주차 빈 값).

In [ ]:
%%sql
-- E2 (1점)
-- [고칠 SQL] 동료가 쓴 SQL — 오류 없이 실행되지만 숫자가 틀립니다. 한 군데를 고쳐 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND phase <> 'PHASE3';

### E3 · 3상 시험 수 다시 세기 (1점)

동료가 원한 것: 3상 제2형 당뇨병 시험은 몇 건인가요? (같은 질문이므로 S2 와 같은 숫자여야 합니다)

💡 힌트: 연구자가 직접 적은 질환 이름(conditions.name)은 표기가 제각각입니다. 표준 질환명(MeSH)은 어느 표 · 어느 열에 있나요? (12주차 오류 10 — 같은 질환, 다른 표기)

In [ ]:
%%sql
-- E3 (1점)
-- [고칠 SQL] 동료가 쓴 SQL — 오류 없이 실행되지만 숫자가 틀립니다. 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE phase = 'PHASE3'
  AND nct_id IN (SELECT nct_id FROM conditions WHERE name = 'Diabetes Mellitus, Type 2');

## 4. 검산표 만들기 — INSERT · UPDATE (10주차)

검산표는 단계마다 건수를 적어 두는 표입니다 (13주차 「검산표」). 이번에는 SQL 로 직접 만듭니다. 내가 만든 연습용 표(TEMP)라 원본 데이터는 그대로입니다.

In [ ]:
%%sql
-- [검산표 준비] 실행만 하세요. 다시 할 때도 이 셀부터 실행합니다
CREATE OR REPLACE TEMP TABLE 검산표 (단계 VARCHAR, 건수 INTEGER);
INSERT INTO 검산표 VALUES ('① 전체 시험', 37509), ('② 제2형 당뇨', 1335);

SELECT * FROM 검산표 ORDER BY 단계;

### T1 · 잘못 적은 건수 고치기 — UPDATE (2점)

동료가 ② 제2형 당뇨의 건수를 1,335 로 잘못 적었습니다. S1 에서 구한 건수로 고치세요. **WHERE 를 빠뜨리면 모든 행이 바뀝니다.**

💡 힌트: 10주차 「UPDATE — 값 바꾸기」 · 「WHERE는 바꿀 대상을 정합니다」. 다시 할 때는 검산표 준비 셀부터 실행하세요.

In [ ]:
%%sql
-- T1 (2점)
-- ___ 를 채워 실행하세요
UPDATE 검산표
SET 건수 = ___
WHERE 단계 = ___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM 검산표 WHERE 단계 IN ('① 전체 시험', '② 제2형 당뇨') ORDER BY 단계;

### T2 · 나머지 단계 넣기 — INSERT (2점)

S2~S4 에서 구한 건수를 검산표에 넣으세요. 단계 이름은 적혀 있고, 건수만 채웁니다.

💡 힌트: 10주차 「INSERT — 새 행 추가하기」. 같은 행이 두 번 들어갔으면 검산표 준비 셀부터 다시 실행하세요.

In [ ]:
%%sql
-- T2 (2점)
-- ___ 를 채워 실행하세요
INSERT INTO 검산표 VALUES
  ('③ 3상', ___),
  ('④ 철회 제외', ___),
  ('⑤ 완료 (분모)', ___),
  ('⑥ 결과 게시 (분자)', ___);

-- 결과 확인 (그대로 두세요)
SELECT * FROM 검산표 WHERE 단계 NOT IN ('① 전체 시험', '② 제2형 당뇨') ORDER BY 단계;

In [ ]:
%%sql
-- 검산표 전체 보기 — 단계마다 몇 건이 빠졌는지 5절 해석에 씁니다
SELECT * FROM 검산표 ORDER BY 단계;

In [ ]:
%%sql
-- 게시율(%) = ⑥ ÷ ⑤ × 100 — 검산표에서 꺼내 계산 (9주차 서브쿼리 · 12주차 오류 6)
SELECT ROUND(100.0 * (SELECT 건수 FROM 검산표 WHERE 단계 = '⑥ 결과 게시 (분자)')
                   / (SELECT 건수 FROM 검산표 WHERE 단계 = '⑤ 완료 (분모)'), 1) AS 게시율;

## 5. 결과 해석 (13주차)

13주차 「결과 해석」 슬라이드의 두 칸을 따라 3~5문장으로 씁니다: 말할 수 있는 것(조건 + 숫자)과 말할 수 없는 것(이유와 함께). 수업 예시(완료된 제2형 당뇨병 전체 시험의 게시율)와 비교해도 좋습니다.

✏️ **해석** (3점, 3~5문장)
→ 

## 제출 확인

In [ ]:
제출확인()

**제출** — 모든 문항이 ✅ 인 상태로 저장(`File → Download → Download .ipynb`)한 뒤 **LMS** 에 제출하세요 (기한 2026-12-08(화)).